# Exercícios extras: Festival ViraBairro

### Monitoria de Extração e Análise de Dados | FGV Comunicação

Dez exercícios de treino para a prova prática, no mesmo caso e com os mesmos nomes de coluna do simulado.

### Como usar

Cada exercício tem quatro partes, nesta ordem: o **enunciado**, uma **célula de código vazia** para você escrever, um **campo de resposta** para a interpretação, e o **gabarito** logo abaixo, recolhido.

**Não abra o gabarito antes de tentar.** Ele está fechado de propósito. Um exercício que você lê a resposta vale zero: o que treina é o desconforto de não saber e ter que procurar. Se travar, vá primeiro no guia de consulta e nos `comandos.md` das aulas.

E escreva a interpretação **sempre**, mesmo quando o código não sair. Metade da nota da prova está nos campos de texto, e é justamente a metade que ninguém treina.

### Sobre os dados

Os arquivos em `dados/` foram construídos para esta monitoria a partir do dicionário do caso. Eles **não são** os arquivos da prova, mas têm a mesma estrutura: mesmos nomes de coluna, mesmas unidades, mesma escala (120 publicações na base de análise, 36 na base bruta), quatro temas e três formatos.

Isso é importante: com 120 publicações repartidas em quatro temas e três formatos, várias contas vão cair em cima de pouquíssimos casos. Isso não é defeito do exercício, é o tipo de situação em que você vai ter que decidir na prova.

### Mapa dos exercícios

| # | O que treina | Questão do simulado | Nível |
|---|---|---|---|
| 1 | Inspeção inicial da base | 1 | fácil |
| 2 | Pipeline de limpeza completo | 2 | médio |
| 3 | Agrupamento e gráfico de barras | 3 | fácil |
| 4 | Tabela por dia, linhas e recorte | 4 | médio |
| 5 | Cruzamento de duas variáveis | 5 | médio |
| 6 | Regressão e comparação de modelos | 7 | médio |
| 7 | Árvore e importância das características | 6 | médio |
| 8 | Classificação e métricas | **8** | difícil |
| 9 | Ajuste do corte de decisão | **8** | difícil |
| 10 | Questão 8 inteira, do zero | **8** | difícil |

Os três últimos são sobre a questão 8. Ela é a mais pesada do simulado e a que tem mais chance de cair, então os exercícios 8 e 9 quebram ela em partes e o 10 junta tudo de novo, do jeito que vai aparecer na prova.

### Antes de começar

Rode a célula abaixo uma vez. Ela tem todos os imports de que você vai precisar nos dez exercícios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

print("pronto")

---

# Exercício 1: diagnóstico da base bruta

`Treina a questão 1` · `nível fácil`

A equipe recebeu `dados/publicacoes_brutas.csv` sem documentação. Antes de usar, faça o diagnóstico. Em uma célula de código, carregue o arquivo e mostre:

1. a quantidade de linhas e colunas, escrita em frase;
2. a quantidade de valores ausentes por coluna, **mostrando somente as colunas que têm ao menos uma ausência**;
3. quantas grafias diferentes a coluna `tema` tem, e quais são elas;
4. o tipo (`dtype`) das colunas `alcance`, `curtidas`, `compartilhamentos` e `salvamentos`.

Na resposta, explique em até duas frases uma limitação que impeça tratar esta base como retrato de todas as redes, públicos ou bairros.

In [ ]:
# Exercício 1
# Carregue dados/publicacoes_brutas.csv.
# Mostre: dimensões, colunas com ausência, grafias de tema, dtypes das colunas de contagem.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e1 = pd.read_csv("dados/publicacoes_brutas.csv")

print(f"A base tem {e1.shape[0]} linhas e {e1.shape[1]} colunas.")

ausentes = e1.isna().sum()
print("\nColunas com ausência:")
print(ausentes[ausentes > 0])          # <- o filtro que o enunciado cobra

print("\nGrafias distintas de tema:", e1["tema"].nunique())
print(sorted(e1["tema"].dropna().unique()))

print("\ndtypes:")
print(e1[["alcance", "curtidas", "compartilhamentos", "salvamentos"]].dtypes)
```

**Saída esperada**

```
A base tem 36 linhas e 12 colunas.

Colunas com ausência:
comentarios    1
salvamentos    1

Grafias distintas de tema: 11
[' cultura', 'CULTURA', 'Cultura', 'MOBILIDADE', 'Mobilidade', 'Saude',
 'TRABALHO', 'Trabalho', 'cultura', 'mobilidade', 'trabalho']

dtypes:
alcance               object
curtidas               int64
compartilhamentos      int64
salvamentos          float64
```

**Três coisas para reparar na saída**

`alcance` veio como `object`, ou seja, texto. Isso acontece porque alguma linha tem um valor que não é número (aqui, `"n/d"`). Uma coluna de contagem que chega como texto é sempre sinal de sujeira.

`salvamentos` veio como `float64` e não como `int64`. Motivo: tem um valor ausente, e `NaN` só existe em float. Tipo float numa coluna de contagem é pista de ausência.

**Onze grafias para quatro temas.** Se você agrupasse por `tema` agora, teria onze grupos em vez de quatro, e todas as suas medianas estariam erradas sem dar erro nenhum.

**Exemplo de resposta**

> A base traz apenas 36 registros de uma única organização, referentes a uma campanha específica, e não tem nenhuma coluna que identifique a rede social, o público ou o bairro alcançado por cada publicação. Com esse tamanho e sem essas informações, os resultados descrevem só este conjunto de publicações e não podem ser lidos como retrato de todas as redes, públicos ou bairros do festival.

Duas frases, como o enunciado pediu. Repare que ela cita limitações **concretas, visíveis na própria base** (36 registros, não existe coluna de rede nem de bairro), em vez de dizer genericamente que os dados são limitados.

</details>

---

# Exercício 2: pipeline de limpeza

`Treina a questão 2` · `nível médio`

Parta de novo de `dados/publicacoes_brutas.csv`, **em uma variável nova**. Faça o tratamento necessário para:

1. remover duplicidade **por `id_publicacao`**;
2. padronizar os valores de `tema` para uma forma consistente;
3. converter `data_publicacao` e as colunas numéricas usadas no cálculo para tipos adequados;
4. tratar ausências ou valores inválidos de maneira justificada;
5. criar `taxa_curtida_pct`, definida por:

$$\frac{curtidas}{alcance} \times 100$$

Depois, produza uma tabela **por formato** com o número de publicações e a **média** de `taxa_curtida_pct`, ordenada da maior para a menor média.

Na resposta, registre as decisões de limpeza em até quatro frases. Se descartar, preencher ou converter algo, diga por quê.

> **Atenção a duas coisas.** A fórmula mudou em relação ao simulado: aqui é só `curtidas`, não `compartilhamentos + salvamentos`. E a tabela pede **média**, não mediana. Ler o enunciado com calma faz parte da prova.

In [ ]:
# Exercício 2
# Recarregue dados/publicacoes_brutas.csv numa variável NOVA.
# Dedup por id, padronize tema, converta tipos, trate ausências e inválidos.
# Crie taxa_curtida_pct e monte a tabela por formato com contagem e MÉDIA.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e2 = pd.read_csv("dados/publicacoes_brutas.csv")
print("linhas no bruto:", len(e2))

# 1. duplicidade POR ID (não por linha inteira)
dups = e2.duplicated(subset="id_publicacao").sum()
e2 = e2.drop_duplicates(subset="id_publicacao", keep="first")
print("duplicadas por id removidas:", dups, "-> restam", len(e2))

# 2. padronizar tema: strip + lower, e só então o dicionário manual para o acento
e2["tema"] = e2["tema"].str.strip().str.lower().replace({"saúde": "saude"})
print("temas:", sorted(e2["tema"].dropna().unique()))

# 3a. data em DUAS passadas (ver a explicação abaixo)
com_ano = e2["data_publicacao"].astype(str).str[:4].str.isdigit().fillna(False)
d1 = pd.to_datetime(e2["data_publicacao"].where(com_ano), errors="coerce")
d2 = pd.to_datetime(e2["data_publicacao"].where(~com_ano),
                    format="mixed", dayfirst=True, errors="coerce")
e2["data_publicacao"] = d1.fillna(d2)
print("período:", e2["data_publicacao"].min(), "até", e2["data_publicacao"].max())
print("datas NaT:", e2["data_publicacao"].isna().sum())

# 3b. só as colunas que entram NESTE cálculo: curtidas e alcance
for c in ["alcance", "curtidas"]:
    e2[c] = pd.to_numeric(e2[c], errors="coerce")

# 4. ausências e inválidos, com motivo
antes = len(e2)
e2 = e2.dropna(subset=["alcance", "curtidas"])   # sem esses dois não dá para calcular a taxa
e2 = e2[e2["alcance"] > 0]                       # alcance zero -> divisão por zero -> inf
e2 = e2[e2["curtidas"] >= 0]                     # contagem negativa é erro de registro
print("descartadas:", antes - len(e2), "-> restam", len(e2))

# 5. a taxa
e2["taxa_curtida_pct"] = e2["curtidas"] / e2["alcance"] * 100

resumo = (
    e2.groupby("formato")
      .agg(publicacoes=("id_publicacao", "count"),
           media_curtida_pct=("taxa_curtida_pct", "mean"))
      .reset_index()
      .sort_values("media_curtida_pct", ascending=False)
      .round(2)
)
resumo
```

**Saída esperada**

```
linhas no bruto: 36
duplicadas por id removidas: 1 -> restam 35
temas: ['cultura', 'mobilidade', 'saude', 'trabalho']
período: 2026-07-06 12:15:00 até 2026-07-20 17:00:00
datas NaT: 1
descartadas: 2 -> restam 33

  formato  publicacoes  media_curtida_pct
     reel           16               5.16
carrossel           14               3.20
   imagem            3               2.70
```

**Por que a data vai em duas passadas**

Esta é a armadilha mais perigosa da prova, porque ela **não dá erro**. A coluna mistura `2026-07-14 11:00` (ano primeiro) com `14/07/2026 11:00` (dia primeiro). Se você aplicar `dayfirst=True` em todas as linhas, o pandas lê o segundo número como dia também nas datas em padrão ISO, e `2026-07-14` vira 7 de... nada, porque 14 não é mês válido, aí vira `NaT`; mas `2026-07-09` viraria 7 de setembro, em silêncio.

O antídoto é o `print` do período. A campanha é de julho e agosto: se aparecer dezembro ali, a conversão leu errado. **Confira `.min()` e `.max()` depois de toda conversão de data.**

**Por que só `alcance` e `curtidas` foram convertidas**

Porque só elas entram nesta fórmula. A linha com `comentarios` ausente **continua na base**, de propósito: essa coluna não participa da conta, e descartar uma publicação inteira por um campo que você não vai usar é jogar dado fora à toa. Dizer isso na resposta mostra critério.

**Exemplo de resposta**

> Removi uma duplicidade por `id_publicacao`, mantendo a primeira ocorrência, e padronizei `tema` com `strip` e `lower`, acrescentando uma tabela de equivalência manual para a grafia que diferia por acento, o que reduziu onze grafias aos quatro temas reais. Converti `data_publicacao` em duas passadas, tratando primeiro as datas com o ano à frente e depois o restante com `dayfirst=True`, porque aplicar `dayfirst` em tudo trocaria dia por mês nas datas em padrão ISO sem gerar erro, e conferi o resultado pelo intervalo entre a menor e a maior data; uma data continuou impossível de interpretar e virou `NaT`. Converti `alcance` e `curtidas` para número com `errors="coerce"`, já que chegaram como texto e são as duas colunas desta fórmula. Descartei as duas linhas em que faltava um desses campos ou em que o alcance era zero, porque sem eles a taxa não pode ser calculada e a divisão por zero produziria infinito, mas mantive a linha com `comentarios` ausente, já que essa coluna não participa do cálculo.

**Um aviso sobre o resultado:** `imagem` aparece com média mais baixa, mas com apenas **3 publicações**. Uma média sobre três casos não decide nada. Se a pergunta pedisse recomendação, isso teria que estar na resposta.

</details>

---

# Exercício 3: qual formato engaja mais

`Treina a questão 3` · `nível fácil`

Use **somente** `dados/publicacoes_analise.csv`, que já vem tratado. Monte uma tabela com o número de publicações e a **mediana** de `taxa_engajamento_pct` **por formato**, ordenada da maior para a menor mediana. Depois faça um gráfico de barras que permita comparar os formatos.

O gráfico deve ter título que comunique a pergunta, eixos nomeados e a indicação "Fonte: dados de treino do Festival ViraBairro (2026)" no próprio gráfico.

Na resposta, escreva de 3 a 5 frases: indique um formato para a equipe priorizar, explique o que a mediana representa e apresente uma limitação. Não afirme causalidade.

In [ ]:
# Exercício 3
# Carregue dados/publicacoes_analise.csv.
# Tabela com contagem e mediana de taxa_engajamento_pct por formato.
# Gráfico de barras com título, eixos nomeados e a fonte dentro do gráfico.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e3 = pd.read_csv("dados/publicacoes_analise.csv")

tabela = (
    e3.groupby("formato")
      .agg(publicacoes=("id_publicacao", "count"),
           mediana_engajamento_pct=("taxa_engajamento_pct", "median"))
      .reset_index()
      .sort_values("mediana_engajamento_pct", ascending=False)
      .round(2)
)
print(tabela.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(tabela["formato"], tabela["mediana_engajamento_pct"], color="#3b6ea5")

ax.set_title("Qual formato tem a maior taxa de engajamento?")   # título = a PERGUNTA
ax.set_xlabel("Formato da publicação")
ax.set_ylabel("Mediana da taxa de engajamento (%)")             # sempre com a unidade

fig.text(0.01, -0.02, "Fonte: dados de treino do Festival ViraBairro (2026)",
         fontsize=8, color="gray")

fig.tight_layout()
plt.show()
```

**Saída esperada**

```
  formato  publicacoes  mediana_engajamento_pct
     reel           49                     7.97
carrossel           47                     5.69
   imagem           24                     4.34
```

**A checklist do gráfico, que vale ponto item por item**

Título que comunica a pergunta (não "Mediana por formato", e sim "Qual formato engaja mais?"). Nome do eixo X. Nome do eixo Y **com a unidade**. E a fonte escrita dentro da figura, não numa célula de texto embaixo.

**Como ler este resultado, e por que ele é diferente do exercício 5**

Aqui as diferenças são grandes e os grupos são grandes: 49, 47 e 24 publicações, com reel quase 2 pontos percentuais acima do carrossel e 3,6 acima da imagem. **Isso é separação de verdade**, e neste caso apontar o primeiro colocado é a resposta certa.

Nem toda tabela é um empate. O que você precisa treinar é olhar os dois números antes de decidir: o tamanho dos grupos e a distância entre eles. Aqui os dois jogam a favor.

**Exemplo de resposta**

> Entre os três formatos, reel apresenta a maior mediana de taxa de engajamento, com 7,97%, contra 5,69% do carrossel e 4,34% da imagem, e é o formato que a equipe deve priorizar. A mediana é o valor do meio da distribuição, metade das publicações acima e metade abaixo, o que a torna menos sensível que a média a uma peça isolada de desempenho atípico. A diferença é consistente e apoiada em grupos razoáveis, 49 reels contra 47 carrosséis e 24 imagens, o que dá mais confiança do que uma separação apertada daria. Como limitação, os formatos não foram distribuídos igualmente entre temas e horários, então parte da vantagem do reel pode vir dessas outras escolhas da equipe. Trata-se de uma associação observada nesta campanha, e não de evidência de que o formato cause mais engajamento.

</details>

---

# Exercício 4: acompanhamento diário e recorte

`Treina a questão 4` · `nível médio`

Use **somente** `dados/publicacoes_analise.csv`, em uma variável nova.

1. Converta `data_publicacao` para data/hora e crie `dia_publicacao`, contendo somente a data.
2. Crie uma tabela por `dia_publicacao` com a quantidade de publicações, a **mediana** de `taxa_engajamento_pct` e o alcance total. Ordene cronologicamente.
3. Faça um gráfico de linhas da mediana de engajamento por dia, com título, eixos nomeados e a fonte no próprio gráfico.
4. Crie um recorte apenas de `carrossel` publicados **antes das 12h**. Mostre as **três** publicações desse recorte com maior `taxa_engajamento_pct`, incluindo `id_publicacao`, `dia_publicacao`, `hora`, `tema` e `taxa_engajamento_pct`.

Na resposta, descreva a variação diária sem afirmar tendência de longo prazo, e explique por que o recorte dos carrosséis matinais não prova que horário ou formato causam engajamento.

In [ ]:
# Exercício 4
# Converta a data, crie dia_publicacao e monte a tabela diária ordenada.
# Gráfico de linhas da mediana por dia.
# Recorte: carrossel E hora < 12. Mostre as três maiores taxas.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e4 = pd.read_csv("dados/publicacoes_analise.csv")

e4["data_publicacao"] = pd.to_datetime(e4["data_publicacao"])
e4["dia_publicacao"] = e4["data_publicacao"].dt.date      # .dt.date corta a hora

tabela_diaria = (
    e4.groupby("dia_publicacao")
      .agg(publicacoes=("id_publicacao", "count"),
           mediana_engajamento_pct=("taxa_engajamento_pct", "median"),
           alcance_total=("alcance", "sum"))
      .reset_index()
      .sort_values("dia_publicacao")        # cronológica = crescente pela data
      .round(2)
)
print("dias com publicação:", len(tabela_diaria))
print("média de publicações por dia:", round(tabela_diaria["publicacoes"].mean(), 2))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(tabela_diaria["dia_publicacao"], tabela_diaria["mediana_engajamento_pct"],
        marker="o", markersize=3, color="#3b6ea5")
ax.set_title("Como a mediana de engajamento variou dia a dia na campanha")
ax.set_xlabel("Dia da publicação")
ax.set_ylabel("Mediana da taxa de engajamento (%)")
ax.tick_params(axis="x", rotation=45)
fig.text(0.01, -0.02, "Fonte: dados de treino do Festival ViraBairro (2026)",
         fontsize=8, color="gray")
fig.tight_layout()
plt.show()

# recorte: DUAS condições, cada uma entre parênteses, ligadas por &
matinais = e4[(e4["formato"] == "carrossel") & (e4["hora"] < 12)]
print("carrosséis antes das 12h:", len(matinais))

top3 = matinais.nlargest(3, "taxa_engajamento_pct")[
    ["id_publicacao", "dia_publicacao", "hora", "tema", "taxa_engajamento_pct"]
]
top3
```

**Saída esperada**

```
dias com publicação: 53
média de publicações por dia: 2.26

carrosséis antes das 12h: 17

id_publicacao dia_publicacao  hora    tema  taxa_engajamento_pct
      OCB-105     2026-08-23    11 cultura                  8.51
      OCB-111     2026-08-25     7   saude                  6.82
      OCB-016     2026-07-14    11 cultura                  6.59
```

**O erro de sintaxe que mais derruba gente aqui**

```python
matinais = e4[e4["formato"] == "carrossel" & e4["hora"] < 12]      # QUEBRA
matinais = e4[(e4["formato"] == "carrossel") and (e4["hora"] < 12)] # QUEBRA
matinais = e4[(e4["formato"] == "carrossel") & (e4["hora"] < 12)]   # certo
```

Cada condição entre parênteses, e `&` no lugar de `and`. Com `and` você leva `The truth value of a Series is ambiguous`, que é uma mensagem que não ajuda em nada a descobrir o que fazer.

**O número que sustenta a interpretação inteira**

120 publicações em 53 dias dá **2,26 por dia**. Uma mediana diária calculada sobre duas publicações praticamente não é uma medida do dia: é quase o valor de uma peça só. Todo o serrilhado do gráfico é, em boa parte, aritmética de amostra pequena.

**Exemplo de resposta**

> A mediana de engajamento oscila bastante de um dia para o outro ao longo dos quase dois meses de campanha, alternando picos e quedas sem uma direção estável de subida ou descida. Boa parte dessa oscilação é aritmética e não comportamento do público: são 120 publicações distribuídas em 53 dias, cerca de duas por dia, e com esse tamanho uma única peça fora do padrão desloca a mediana do dia inteiro. Por isso o gráfico não autoriza nenhuma afirmação sobre tendência de longo prazo, apenas sobre variação diária. O recorte dos três carrosséis matinais de maior taxa também não prova que horário ou formato causem engajamento, porque ele seleciona os melhores casos de um grupo de apenas 17 publicações que já havia sido filtrado justamente por formato e horário: os casos foram escolhidos pelo resultado que se quer explicar. Para sustentar que carrossel matinal funciona seria preciso comparar carrosséis matinais com carrosséis de outros horários e com outros formatos no mesmo horário, e essa comparação não foi feita.

</details>

---

# Exercício 5: tema e formato juntos

`Treina a questão 5` · `nível médio`

Use **somente** `dados/publicacoes_analise.csv`. Crie uma tabela com **uma linha para cada combinação** de `tema` e `formato`, contendo o número de publicações e a **média** de `taxa_engajamento_pct`, ordenada da maior para a menor média. Depois faça um gráfico de barras que compare as combinações, com título, eixos nomeados e a fonte no próprio gráfico.

Na resposta, escreva de 4 a 6 frases: destaque uma combinação que mereça ser testada, explique por que comparar duas variáveis é diferente de analisar apenas uma, e registre uma limitação da base.

> **Dica de leitura:** depois de montar a tabela ordenada por média, reorganize ela mentalmente **por tema** e veja se aparece algum padrão que a ordenação por média estava escondendo.

In [ ]:
# Exercício 5
# Agrupe por tema E formato (lista no groupby), com contagem e MÉDIA.
# Ordene da maior para a menor média e faça o gráfico das combinações.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e5 = pd.read_csv("dados/publicacoes_analise.csv")

# groupby com DUAS chaves: basta passar uma lista
tabela = (
    e5.groupby(["tema", "formato"])
      .agg(publicacoes=("id_publicacao", "count"),
           media_engajamento_pct=("taxa_engajamento_pct", "mean"))
      .reset_index()
      .sort_values("media_engajamento_pct", ascending=False)
      .round(2)
)
print(tabela.to_string(index=False))

tabela["combinacao"] = tabela["tema"] + " / " + tabela["formato"]

fig, ax = plt.subplots(figsize=(9, 6))
# barh porque o rótulo é comprido; [::-1] põe a maior no topo
ax.barh(tabela["combinacao"][::-1], tabela["media_engajamento_pct"][::-1], color="#3b6ea5")
ax.set_title("Quais combinações de tema e formato engajam mais?")
ax.set_xlabel("Média da taxa de engajamento (%)")   # no barh, o valor fica no eixo X
ax.set_ylabel("Tema / formato")
fig.text(0.01, -0.02, "Fonte: dados de treino do Festival ViraBairro (2026)",
         fontsize=8, color="gray")
fig.tight_layout()
plt.show()
```

**Saída esperada**

```
      tema   formato  publicacoes  media_engajamento_pct
   cultura      reel           26                   8.86
     saude      reel            8                   7.12
mobilidade      reel            9                   6.87
   cultura carrossel           20                   6.85
  trabalho      reel            6                   5.91
   cultura    imagem            7                   5.76
     saude carrossel            8                   5.46
     saude    imagem            4                   4.65
mobilidade carrossel           12                   4.50
  trabalho carrossel            7                   4.09
mobilidade    imagem            2                   4.03
  trabalho    imagem           11                   3.25
```

**O padrão que a ordenação esconde**

Reorganize por tema e olhe:

| tema | reel | carrossel | imagem |
|---|---|---|---|
| cultura | 8,86 | 6,85 | 5,76 |
| saude | 7,12 | 5,46 | 4,65 |
| mobilidade | 6,87 | 4,50 | 4,03 |
| trabalho | 5,91 | 4,09 | 3,25 |

**A ordem reel, carrossel, imagem se repete nos quatro temas, sem uma única exceção.** Esse é o achado de verdade do exercício, e ele não aparece na tabela ordenada por média: só aparece quando você reagrupa. É esse tipo de leitura, que reorganiza a tabela para enxergar estrutura em vez de ler a primeira linha, que separa uma resposta de nota máxima das outras.

**E a armadilha do topo**

`mobilidade / imagem` tem **2 publicações**. `trabalho / reel` tem 6. Uma média sobre dois casos é literalmente a média de duas peças: uma delas sendo atípica, o valor inteiro se move. A coluna `publicacoes` não é decoração, é ela que diz em qual linha confiar.

**Exemplo de resposta**

> A combinação de cultura em reel aparece no topo, com média de 8,86%, e é a mais segura de recomendar, porque além de liderar ela é também a combinação com mais publicações da tabela, 26, o que dá alguma estabilidade ao número. O cruzamento revela um padrão que a análise de uma variável isolada esconderia: dentro de cada um dos quatro temas, sem exceção, a ordem é reel, depois carrossel, depois imagem, o que indica que o formato ordena o resultado de forma consistente. É por isso que comparar duas variáveis juntas é diferente de analisar cada uma separadamente: olhando só o tema, a vantagem de cultura poderia vir apenas de ela concentrar mais reels, e olhando só o formato, perderíamos o fato de que a vantagem do reel se mantém independentemente do tema. Como limitação, várias células da tabela têm pouquíssimos casos, com mobilidade em imagem chegando a apenas 2 publicações e trabalho em reel a 6, e médias calculadas sobre esse número são instáveis o bastante para trocar de posição por causa de uma única peça atípica. Por isso trato a metade de baixo da tabela como indicação preliminar, não como resultado. A recomendação que os dados realmente sustentam é priorizar reel como formato, decisão que se repete em todos os temas, e tratar a escolha do tema como decisão editorial.

</details>

---

# Exercício 6: estimar a taxa antes de publicar

`Treina a questão 7` · `nível médio`

A equipe quer estimar, **antes da publicação**, a `taxa_engajamento_pct` esperada de uma nova peça. Use **somente** `dados/publicacoes_analise.csv`.

1. Na resposta, indique o tipo de aprendizado adequado (regressão, classificação ou clusterização) e explique por que a variável-alvo exige essa escolha.
2. Use como características somente `tema`, `formato`, `seguidores_autor`, `videos_autor`, `tamanho_legenda`, `n_emojis`, `n_hashtags`, `hora`, `dia_semana` e `duracao_segundos`. Transforme categorias em números quando necessário.
3. Reserve 75% para treino e 25% para teste, com `random_state=42`.
4. Ajuste uma regressão linear e uma árvore de regressão com **profundidade máxima 3**. Compare os dois numa tabela de MAE e R², acrescentando um modelo que sempre chuta a média do treino.
5. Faça um gráfico de dispersão entre valores reais e previstos para o modelo de menor MAE, com uma linha de referência onde previsão e valor real seriam iguais.

Na resposta, explique o que o MAE mede, indique o modelo escolhido e registre uma limitação que impeça interpretar a previsão como relação causal.

In [ ]:
# Exercício 6
# get_dummies nas categóricas, alvo = taxa_engajamento_pct.
# train_test_split SEM stratify (alvo contínuo).
# Linear, árvore prof. 3 e modelo bobo. Tabela de MAE e R2. Dispersão com diagonal.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e6 = pd.read_csv("dados/publicacoes_analise.csv")

CARACTERISTICAS = ["tema", "formato", "seguidores_autor", "videos_autor", "tamanho_legenda",
                   "n_emojis", "n_hashtags", "hora", "dia_semana", "duracao_segundos"]

# A LINHA QUE SALVA A PROVA: texto vira coluna de 0 e 1
X = pd.get_dummies(e6[CARACTERISTICAS], columns=["tema", "formato"])
y = e6["taxa_engajamento_pct"]
print("colunas do X:", X.shape[1])

# SEM stratify: o alvo é contínuo
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=42)
print("treino:", len(y_treino), "| teste:", len(y_teste))

linear = LinearRegression().fit(X_treino, y_treino)
previsao_linear = linear.predict(X_teste)

arvore = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_treino, y_treino)
previsao_arvore = arvore.predict(X_teste)

previsao_boba = np.full(len(y_teste), y_treino.mean())   # o baseline

comparacao = pd.DataFrame({
    "modelo": ["modelo bobo (média)", "regressão linear", "árvore de regressão (prof. 3)"],
    "MAE": [mean_absolute_error(y_teste, p) for p in (previsao_boba, previsao_linear, previsao_arvore)],
    "R2":  [r2_score(y_teste, p) for p in (previsao_boba, previsao_linear, previsao_arvore)],
}).round(3)
print(comparacao.to_string(index=False))

# dispersão do modelo de menor MAE
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(y_teste, previsao_linear, alpha=0.6, s=25, color="#3b6ea5")
minimo = float(min(y_teste.min(), previsao_linear.min()))
maximo = float(max(y_teste.max(), previsao_linear.max()))
ax.plot([minimo, maximo], [minimo, maximo], color="#c0392b",
        linestyle="--", label="previsão = valor real")
ax.set_title("Valor real x valor previsto (regressão linear)")
ax.set_xlabel("Taxa de engajamento real (%)")
ax.set_ylabel("Taxa de engajamento prevista (%)")
ax.legend()
fig.tight_layout()
plt.show()
```

**Saída esperada**

```
colunas do X: 15
treino: 90 | teste: 30

                       modelo   MAE     R2
          modelo bobo (média) 1.654 -0.006
             regressão linear 0.995  0.551
árvore de regressão (prof. 3) 1.194  0.436
```

**Três coisas para reparar**

**São 15 colunas, não 16.** Quatro temas mais três formatos dão sete colunas novas, e as três categóricas originais somem. Se você esperava `formato_video`, não existe: esta base tem três formatos.

**Aqui não tem `stratify`.** Ele serve para manter proporção entre categorias, e o alvo aqui é contínuo. Se você copiar o `train_test_split` de um exercício de classificação com o `stratify=y` junto, o scikit-learn quebra.

**A linear ganhou da árvore**, o que é o oposto do que aconteceu na aula 11. Não existe modelo que seja sempre melhor: com 90 publicações de treino, a árvore não tem dados suficientes para aproveitar a flexibilidade que teria. Você compara e escolhe pelo número, sempre.

**Como ler a diagonal:** ponto acima da linha significa que o modelo previu mais do que aconteceu, abaixo significa que previu menos. Nuvem larga significa erro grande.

**Exemplo de resposta**

> O tipo de aprendizado adequado é a regressão, porque a variável-alvo, `taxa_engajamento_pct`, é um número contínuo e a equipe quer estimar o valor esperado dessa taxa, não classificá-la numa categoria. A classificação responderia a uma pergunta diferente, do tipo "passa ou não passa de determinado patamar", e ainda exigiria um corte arbitrário; a clusterização não se aplica porque não há alvo a prever, já que ela apenas agrupa registros por semelhança sem usar resposta conhecida. O MAE mede o erro absoluto médio na mesma unidade do alvo: o MAE de 0,995 da regressão linear significa que, em média, a previsão erra a taxa de engajamento em cerca de um ponto percentual, para cima ou para baixo. Escolhi a regressão linear, que teve o menor MAE, 0,995 contra 1,194 da árvore, e o maior R², 0,551 contra 0,436; os dois superam o modelo que sempre chuta a média, com MAE 1,654, o que indica que existe padrão real sendo aprendido. Como limitação, o modelo descreve associações observadas nesta campanha específica e não relações causais: as características não foram distribuídas de forma controlada entre as publicações, então alterar uma delas numa nova peça não garante o efeito que o modelo estima.

</details>

---

# Exercício 7: o que a árvore mais usou

`Treina a questão 6` · `nível médio`

Use **somente** `dados/publicacoes_analise.csv` e refaça esta questão de modo independente.

1. Crie `destaque` pelo **percentil 80** de `taxa_engajamento_pct`.
2. Use apenas as mesmas características permitidas do exercício anterior, transformando as categorias em números.
3. Reserve 75% para treino e 25% para teste, com `random_state=42` e **preservando a proporção do alvo**. Ajuste uma árvore de classificação com **profundidade máxima 3**, `random_state=42` e **pesos balanceados**.
4. Produza uma tabela ordenada e um gráfico de barras horizontal com as cinco características de maior importância.

Na resposta, explique por que uma importância alta não prova causalidade e descreva uma mudança na base que poderia alterar esse ranking.

> **Cuidado:** o percentil aqui é **80**, não 75. E a profundidade é **3**, não 4.

In [ ]:
# Exercício 7
# Alvo pelo percentil 80. get_dummies. Split COM stratify.
# Árvore max_depth=3, random_state=42, class_weight="balanced".
# Top 5 importâncias em tabela e em barh.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e7 = pd.read_csv("dados/publicacoes_analise.csv")

corte = e7["taxa_engajamento_pct"].quantile(0.80)          # percentil 80
e7["destaque"] = (e7["taxa_engajamento_pct"] > corte).astype(int)
print(f"corte p80: {corte:.2f} | positivos: {e7['destaque'].sum()} de {len(e7)}")

CARACTERISTICAS = ["tema", "formato", "seguidores_autor", "videos_autor", "tamanho_legenda",
                   "n_emojis", "n_hashtags", "hora", "dia_semana", "duracao_segundos"]
X = pd.get_dummies(e7[CARACTERISTICAS], columns=["tema", "formato"])
y = e7["destaque"]

# COM stratify: o alvo agora é categoria
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("treino:", len(y_treino), f"({int(y_treino.sum())} positivos)",
      "| teste:", len(y_teste), f"({int(y_teste.sum())} positivos)")

arvore = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight="balanced")
arvore.fit(X_treino, y_treino)

importancias = (pd.DataFrame({"caracteristica": X.columns,
                              "importancia": arvore.feature_importances_})
                  .sort_values("importancia", ascending=False)
                  .reset_index(drop=True))
top5 = importancias.head(5)
print(top5.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top5["caracteristica"][::-1], top5["importancia"][::-1], color="#3b6ea5")
ax.set_title("O que a árvore mais usou para separar as publicações de destaque")
ax.set_xlabel("Importância na árvore (0 a 1)")
ax.set_ylabel("Característica")
fig.tight_layout()
plt.show()
```

**Saída esperada**

```
corte p80: 8.37 | positivos: 23 de 120
treino: 90 (17 positivos) | teste: 30 (6 positivos)

  caracteristica  importancia
    tema_cultura     0.613774
duracao_segundos     0.192028
            hora     0.135655
      n_hashtags     0.058543
    videos_autor     0.000000
```

**Repare na quinta linha: importância zero.**

`videos_autor` aparece no "top 5" com importância **0,000**. Isso não é bug: uma árvore de profundidade 3 faz no máximo sete divisões, então ela usa pouquíssimas características e todas as outras ficam com zero. O `head(5)` pega as cinco primeiras da lista ordenada, e se só quatro tiverem importância acima de zero, a quinta vem zerada.

Se o enunciado pede as cinco maiores, entregue as cinco. Mas **mencione isso na resposta**: dizer que a árvore na prática usou só quatro características é uma observação que mostra que você entendeu o que está olhando, em vez de só ter rodado o comando.

**Por que o percentil 80 muda o problema**

Com p75 sobrariam 30 positivos; com p80 sobram 23, e no conjunto de teste restam **6**. Quanto mais alto o corte, mais rara a classe e mais instável qualquer métrica. É por isso que o enunciado sempre manda conferir quantos positivos sobraram.

**Exemplo de resposta**

> A árvore usou principalmente `tema_cultura`, com importância 0,61, seguido de `duracao_segundos` com 0,19, `hora` com 0,14 e `n_hashtags` com 0,06; a quinta característica do ranking aparece com importância zero, o que indica que uma árvore de profundidade 3 acabou usando apenas quatro características para separar as publicações. Importância alta significa apenas que a característica foi útil para dividir os dados que a árvore viu no treino, não que ela cause o desempenho: `tema_cultura` pode estar no topo porque as publicações de cultura desta campanha concentram determinados formatos e horários, e a árvore aproveita essa associação sem conseguir separar o que vem de quê. A limitação decisiva é o tamanho da base: a árvore foi treinada com 90 publicações, das quais apenas 17 pertencem à classe positiva, e com esse número o ranking é frágil. Uma mudança pequena já o alteraria: trocar o `random_state` do `train_test_split`, aumentar a profundidade máxima da árvore ou acrescentar uma dezena de publicações mudaria quais características entram nas primeiras divisões e, com isso, todo o ranking, sem que nada tivesse mudado no fenômeno real.

</details>

---
---

# Bloco final: a questão 8

As três atividades a seguir são todas sobre a questão 8 do simulado. Ela é a mais pesada da prova e a com maior chance de cair, então vale treinar com calma.

A estratégia é esta: o **exercício 8** treina a montagem e a leitura das métricas, o **exercício 9** treina só o ajuste do corte, e o **exercício 10** junta tudo, do zero, no formato exato em que a questão aparece.

Se você só tiver tempo para uma coisa antes da prova, faça o exercício 10 cronometrado.

---

# Exercício 8: comparar classificadores

`Treina a questão 8` · `nível difícil`

Nos dias finais da campanha, a equipe só consegue dar divulgação adicional a poucas publicações. Use **somente** `dados/publicacoes_analise.csv`.

1. Crie `mereceu_divulgacao_adicional`: 1 quando `taxa_engajamento_pct` estiver **acima do percentil 75** da própria base, 0 nos demais.
2. Use apenas as características disponíveis antes da publicação, transformando categorias em colunas numéricas. Não use identificador, alcance, interações nem a própria taxa: isso seria vazamento.
3. Reserve 75% para treino e 25% para teste, com `random_state=42` e **preservando a proporção** do alvo nos dois conjuntos.
4. Compare estes três classificadores: **regressão logística, Random Forest e Gaussian Naive Bayes**.
5. Apresente precisão, recall e F1 dos três numa tabela, e exiba a matriz de confusão do modelo com maior F1.

Na resposta, diga qual modelo você escolheria e explique o que falso positivo e falso negativo significam para a equipe de comunicação.

> **Os imports que talvez você não conheça:**
> ```python
> from sklearn.ensemble import RandomForestClassifier
> from sklearn.naive_bayes import GaussianNB
> ```
> Os dois usam `fit` e `predict` exatamente como a logística e a árvore. Trocar de modelo é trocar uma linha.

In [ ]:
# Exercício 8
# Alvo pelo percentil 75. get_dummies nas categóricas.
# Split 75/25 com random_state=42 e stratify.
# Compare logística, Random Forest e Naive Bayes: precisão, recall, F1.
# Matriz de confusão do modelo com maior F1.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
e8 = pd.read_csv("dados/publicacoes_analise.csv")

corte = e8["taxa_engajamento_pct"].quantile(0.75)
e8["mereceu_divulgacao_adicional"] = (e8["taxa_engajamento_pct"] > corte).astype(int)
print(f"corte p75: {corte:.2f} | positivos: {e8['mereceu_divulgacao_adicional'].sum()} de {len(e8)}")

CARACTERISTICAS = ["tema", "formato", "seguidores_autor", "videos_autor", "tamanho_legenda",
                   "n_emojis", "n_hashtags", "hora", "dia_semana", "duracao_segundos"]
X = pd.get_dummies(e8[CARACTERISTICAS], columns=["tema", "formato"])
y = e8["mereceu_divulgacao_adicional"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("teste:", len(y_teste), "publicações,", int(y_teste.sum()), "positivas")

logistica = LogisticRegression(max_iter=5000)
modelos = {
    "regressão logística": logistica,
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
}

linhas = []
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    decisao = modelo.predict(X_teste)
    linhas.append({"modelo": nome,
                   "precisão": precision_score(y_teste, decisao, zero_division=0),
                   "recall": recall_score(y_teste, decisao, zero_division=0),
                   "F1": f1_score(y_teste, decisao, zero_division=0)})

resultado = pd.DataFrame(linhas).sort_values("F1", ascending=False).round(3)
print(resultado.to_string(index=False))

melhor = modelos[resultado.iloc[0]["modelo"]]
cm = confusion_matrix(y_teste, melhor.predict(X_teste))
print(f"\nmatriz de confusão do {resultado.iloc[0]['modelo']}:")
print(f"  VN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  VP={cm[1,1]}")
```

**Saída esperada**

```
corte p75: 7.98 | positivos: 30 de 120
teste: 30 publicações, 7 positivas

             modelo  precisão  recall    F1
      Random Forest     1.000   0.714 0.833
regressão logística     0.500   0.286 0.364
        Naive Bayes     0.222   0.286 0.250

matriz de confusão do Random Forest:
  VN=23  FP=0
  FN=2  VP=5
```

**O resultado mais importante deste exercício**

**A regressão logística foi mal.** F1 de 0,364, pegando 2 das 7 publicações que mereciam apoio. O Random Forest ganhou com folga.

Isso importa porque, nos materiais da disciplina e no simulado, a logística costuma se sair bem, e é tentador decorar "a logística ganha". **Não decore.** Você compara os três e escreve o nome do que está na primeira linha da sua tabela, qualquer que seja ele.

**E olhe a coluna de precisão do Random Forest: 1,000.**

Precisão 1,000 significa que **ele nunca errou quando apostou**: das 5 publicações que marcou como merecedoras, as 5 mereciam mesmo. Parece perfeito. Mas o recall é 0,714: das 7 que realmente mereciam, ele encontrou 5 e deixou 2 passarem.

Um modelo que só aposta quando tem certeza absoluta tem precisão alta e deixa oportunidade na mesa. É exatamente por isso que precisão sozinha não basta, e é exatamente o problema que o exercício 9 vai atacar mexendo no corte.

**Sobre o `ConvergenceWarning`**

A logística provavelmente vai soltar um aviso amarelo dizendo que não convergiu. É porque as características estão em escalas muito diferentes (seguidores na casa dos milhares, hora de 0 a 23). **É aviso, não erro: o modelo rodou e os resultados valem.**

**Exemplo de resposta**

> Escolheria o Random Forest, que teve o maior F1 entre os três modelos, com 0,833, contra 0,364 da regressão logística e 0,250 do Naive Bayes. Um falso positivo significa gastar um dos poucos espaços de divulgação adicional numa publicação que não iria render, desperdiçando um recurso escasso nos dias finais da campanha, e um falso negativo significa deixar sem apoio uma peça que teria bom desempenho, uma oportunidade que não volta antes do festival. A matriz de confusão mostra que o Random Forest não cometeu nenhum falso positivo, acertando as 5 publicações em que apostou, mas deixou passar 2 das 7 que mereciam apoio. Ou seja, ele erra a favor da economia de recursos e contra o aproveitamento de oportunidades, o que pode não ser o que a equipe quer faltando duas semanas para o festival. Vale registrar que o conjunto de teste tem apenas 30 publicações e 7 positivas, de modo que cada acerto ou erro move o recall em cerca de 0,14 ponto e a comparação entre os modelos é menos estável do que os números sugerem.

</details>

---

# Exercício 9: ajustar o corte de decisão

`Treina a questão 8` · `nível difícil`

Continuando do exercício anterior, e usando a **regressão logística já ajustada** no mesmo conjunto de teste:

1. Obtenha a probabilidade de cada publicação de teste pertencer à classe 1.
2. Compare os cortes **0,50, 0,40, 0,30 e 0,20** para decidir quando uma publicação recebe divulgação adicional.
3. Apresente numa tabela, para cada corte: precisão, recall, F1, e as contagens de acertos (VP), alarmes falsos (FP) e publicações que passaram batido (FN).

Na resposta, indique qual corte você escolheria e justifique com base no que observou, em até quatro frases.

> **A ferramenta que você precisa:** `.predict()` decide sempre com corte fixo em 0,50. Para mexer no corte, use `.predict_proba(X_teste)[:, 1]`, que devolve a probabilidade da classe 1, e compare você mesmo contra o valor que quiser.

In [ ]:
# Exercício 9
# Pegue a probabilidade da classe 1 na logística do exercício anterior.
# Para cada corte em [0.50, 0.40, 0.30, 0.20], calcule precisão, recall, F1, VP, FP e FN.
# Monte a tabela comparativa.



**Sua resposta:**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

```python
# reaproveita a logistica, X_teste e y_teste do exercício 8
probabilidade = logistica.predict_proba(X_teste)[:, 1]   # coluna 1 = prob da classe 1

linhas = []
for corte in [0.50, 0.40, 0.30, 0.20]:
    decisao = (probabilidade >= corte).astype(int)       # eu decido, não o .predict()
    linhas.append({
        "corte": corte,
        "precisão": precision_score(y_teste, decisao, zero_division=0),
        "recall": recall_score(y_teste, decisao, zero_division=0),
        "F1": f1_score(y_teste, decisao, zero_division=0),
        "VP": int(((decisao == 1) & (y_teste.values == 1)).sum()),
        "FP": int(((decisao == 1) & (y_teste.values == 0)).sum()),
        "FN": int(((decisao == 0) & (y_teste.values == 1)).sum()),
    })

pd.DataFrame(linhas).round(3)
```

**Saída esperada**

```
 corte  precisão  recall    F1  VP  FP  FN
   0.5     0.500   0.286 0.364   2   2   5
   0.4     0.429   0.429 0.429   3   4   4
   0.3     0.400   0.571 0.471   4   6   3
   0.2     0.500   0.857 0.632   6   6   1
```

**Leia a tabela de baixo para cima**

No corte padrão de 0,50, a logística encontra **2 das 7** publicações que mereciam apoio e deixa **5 passarem**. É um modelo praticamente inútil para o que a equipe precisa.

Conforme o corte cai, o recall sobe de 0,286 para 0,857: no corte 0,20 ele encontra 6 das 7. E, diferente do roteiro que se costuma decorar, **a precisão não despencou**: ela cai até 0,40 e depois **volta para 0,50** no corte mais baixo. O F1 sobe o tempo todo.

**Por que isso contraria o roteiro padrão**

A regra geral "baixar o corte aumenta o recall e derruba a precisão" vale na média, mas não é uma lei. Aqui o corte padrão de 0,50 estava simplesmente mal posicionado para esta base: o modelo atribuía probabilidades baixas até para publicações que mereciam, então quase nunca apostava.

A lição prática: **0,50 não tem nada de sagrado.** Ele é o padrão do `.predict()`, não uma escolha analítica. Testar outros cortes é barato e às vezes, como aqui, melhora tudo ao mesmo tempo.

**A ressalva que não pode faltar**

O conjunto de teste tem **7 positivos**. Cada publicação que o modelo acerta ou erra move o recall em cerca de **0,143**. Toda a diferença entre o corte 0,30 e o 0,20 são duas publicações. Essa tabela indica uma direção, não estabelece uma regra.

**Exemplo de resposta**

> Escolheria o corte de 0,20, que é o que apresenta o maior F1 da tabela, 0,632, e que encontra 6 das 7 publicações que realmente mereciam divulgação adicional, contra apenas 2 no corte padrão de 0,50. Neste caso não houve o trade-off habitual entre precisão e recall: a precisão no corte 0,20 é de 0,50, a mesma do corte 0,50, de modo que baixar o corte melhorou o recall sem custo líquido de precisão, o que indica que o corte padrão estava mal posicionado para esta base. A escolha também se justifica pelo contexto, porque faltando duas semanas para o festival deixar de apoiar uma peça promissora é um erro irreversível, enquanto um alarme falso custa um espaço de divulgação que pode ser remanejado. Como ressalva, o conjunto de teste tem apenas 7 publicações positivas, de modo que cada acerto move o recall em cerca de 0,14 ponto e a vantagem observada não deveria ser tratada como regra estável para as próximas campanhas.

</details>

---

# Exercício 10: a questão 8 inteira, do zero

`Treina a questão 8` · `nível difícil` · **cronometre 35 minutos**

Este é o exercício mais importante do caderno. Feche os exercícios 8 e 9, abra uma célula limpa e faça a questão completa, do jeito que ela vai aparecer na prova.

---

Nos dias finais da campanha, a equipe só consegue dar divulgação adicional a poucas publicações. Antes de uma nova publicação ir ao ar, a pergunta é: quais peças têm maior chance de receber a classificação **"merece divulgação adicional"**?

Use **somente** `dados/publicacoes_analise.csv`. Crie `mereceu_divulgacao_adicional`: valor 1 quando `taxa_engajamento_pct` estiver acima do percentil 75 da própria base, e 0 nos demais.

1. Na resposta, diga qual tipo de aprendizado de máquina é adequado (regressão, classificação ou clusterização) e explique brevemente por que os outros dois não respondem diretamente a essa decisão.
2. Use apenas características disponíveis antes da publicação: `tema`, `formato`, `seguidores_autor`, `videos_autor`, `tamanho_legenda`, `n_emojis`, `n_hashtags`, `hora`, `dia_semana` e `duracao_segundos`. Transforme as categorias em colunas numéricas quando necessário. Não use identificador, alcance, interações ou `taxa_engajamento_pct`: isso seria vazamento de dados.
3. Reserve 75% das publicações para treino e 25% para teste, usando `random_state=42` e preservando a proporção de publicações que mereceram divulgação adicional nos dois conjuntos.
4. Escolha três classificadores desta lista e compare-os: regressão logística, árvore de classificação, Random Forest, Extra Trees, AdaBoost e Gaussian Naive Bayes.
5. No conjunto de teste, apresente precisão, recall e F1 dos três modelos em uma tabela e exiba a matriz de confusão do modelo com maior F1.
6. Na regressão logística já ajustada e no mesmo conjunto de teste, compare os cortes 0,50 e 0,30. Apresente precisão, recall e F1 dos dois cortes em uma tabela.

Na resposta final, informe o modelo e o corte escolhidos. Explique, em até seis frases, o que falso positivo e falso negativo significam para a equipe de comunicação e justifique sua decisão com base no trade-off observado entre precisão e recall.

In [ ]:
# Exercício 10: questão 8 completa
# Faça tudo aqui, do zero, sem olhar os exercícios 8 e 9.



**Sua resposta 10.1 (tipo de modelo):**

*(escreva aqui)*

&nbsp;

**Sua resposta 10 (decisão e riscos de erro):**

*(escreva aqui)*

&nbsp;

&nbsp;

<details>
<summary><b>GABARITO</b> (clique para abrir só depois de tentar)</summary>

O código é a junção dos exercícios 8 e 9, com a lista de cortes reduzida a `[0.50, 0.30]`. Em vez de repetir, aqui vai a **checklist de correção**: é o que um corretor procura, item por item.

**Código (cada item costuma valer ponto)**

- [ ] Carregou a base numa variável nova, sem depender de outra questão.
- [ ] Criou o alvo com `quantile(0.75)` e comparação `>` (acima do percentil, não a partir dele).
- [ ] Usou **exatamente** as dez características listadas, sem acrescentar nem tirar.
- [ ] Aplicou `pd.get_dummies` em `tema` e `formato`.
- [ ] Não colocou `alcance`, interações, `taxa_engajamento_pct` nem `id_publicacao` no `X`.
- [ ] `test_size=0.25`, `random_state=42` e `stratify=y`, os três.
- [ ] Comparou **três** modelos da lista, num único quadro com precisão, recall e F1.
- [ ] Exibiu a matriz de confusão **do modelo com maior F1**, não de um modelo qualquer.
- [ ] Usou `predict_proba(...)[:, 1]` para os cortes, não `predict()`.
- [ ] Apresentou os dois cortes em tabela, não em texto solto.

**Resposta escrita (é onde está metade da nota)**

- [ ] Disse o tipo de aprendizado **e** por que os outros dois não servem, um de cada vez.
- [ ] Nomeou o modelo escolhido citando o **número** que sustenta a escolha.
- [ ] Explicou falso positivo e falso negativo **em termos da equipe de comunicação** (espaço de divulgação desperdiçado, oportunidade perdida), não em termos de definição de livro.
- [ ] Nomeou o corte escolhido e justificou com os números das duas linhas da tabela.
- [ ] Contou as frases. O enunciado diz "até seis".
- [ ] Registrou pelo menos uma limitação, e a mais forte disponível é o tamanho do conjunto de teste.

**Os números que devem sair**

Corte p75 em 7,98, com 30 positivos em 120. Teste com 30 publicações e 7 positivas. Random Forest com F1 0,833 (precisão 1,000 e recall 0,714) na frente da logística, com 0,364. No ajuste de corte da logística, 0,50 dá precisão 0,500 e recall 0,286; 0,30 dá precisão 0,400 e recall 0,571.

**Os três erros que mais aparecem nesta questão**

**Esquecer o `get_dummies`.** O código quebra com `could not convert string to float` e a questão inteira vai embora. É o erro número um.

**Passar `stratify=y` numa questão de regressão ou esquecer dele aqui.** Aqui ele é obrigatório: o enunciado diz "preservando a proporção".

**Escrever a resposta sem olhar a tabela.** Decorar "a logística ganha e baixar o corte derruba a precisão" é o caminho mais rápido para escrever algo que os seus próprios números desmentem. Neste exercício, quem decorou erraria as duas coisas.

**Se você travou no código, escreva a resposta mesmo assim.** Explique o que faria e por quê. Campo de texto em branco vale zero; campo com raciocínio correto e código quebrado ainda vale bastante.

</details>

---
---

# Fechamento

### Se você acertou tudo

Refaça o exercício 10 cronometrado, com o caderno fechado. Saber fazer e saber fazer em 35 minutos com alguém olhando são coisas diferentes.

### Se você travou em algum

Anote **onde** travou, não só que travou. Foi na sintaxe, foi em não saber qual comando usar, ou foi em saber o comando e não saber o que responder depois? Cada um desses tem um remédio diferente, e só o primeiro se resolve relendo código.

### Os seis erros que mais custaram ponto neste caderno

1. **Esquecer `pd.get_dummies`** nos exercícios 6, 7, 8 e 10.
2. **Usar `stratify` em regressão**, ou esquecer dele em classificação.
3. **Usar `drop_duplicates()` sem `subset="id_publicacao"`** quando o enunciado pede por id.
4. **Jogar `dayfirst=True`** numa coluna de datas em formatos misturados, sem conferir `.min()` e `.max()` depois.
5. **Gráfico sem título, sem nome de eixo ou sem a fonte.**
6. **Responder o campo de texto em uma frase genérica**, ou deixar em branco.

### As duas perguntas que resolvem quase toda interpretação

Antes de recomendar qualquer coisa a partir de um número:

**Quantos casos tem por trás dele?** Uma média sobre 2 publicações, como apareceu no exercício 5, não é uma medida.

**Qual é a distância para o segundo colocado?** Se for menor do que o deslocamento que uma peça atípica causaria, você não tem um vencedor, tem um empate. E dizer isso, em vez de apontar a barra mais alta, é o que separa a nota boa da nota mediana.

Boa prova. Qualquer dúvida, me chamem **antes**, não depois.